# 70. ACE/EPAM Implementation / ACE/EPAM 구현

**Paper**: Gold, R. E., Krimigis, S. M., Hawkins, S. E. III, Haggerty, D. K., Lohr, D. A., Fiore, E., Armstrong, T. P., Holland, G., Lanzerotti, L. J. (1998), *Electron, Proton, and Alpha Monitor on the Advanced Composition Explorer Spacecraft*, Space Science Reviews 86, 541-562. DOI: 10.1023/A:1005088115759

---

**English.** This notebook implements four core EPAM concepts using synthetic data:
1. The 5 SSD telescope angular geometry (LEMS30, LEMS120, LEFS60, LEFS150, CA60) with sector look directions on the unit sphere.
2. Electron / ion separation by energy and stopping power ($dE/dx$) — the physical basis of LEMS (magnet) vs LEFS (foil) design and the $\Delta E \times E$ ion-species identification in CA60.
3. Near-real-time SEP onset detection using a $4\sigma$-above-1-day-baseline trigger — the operational logic behind NOAA's RTSW SEP alerts that EPAM has supported since 1998.
4. First-order anisotropy from sector counts and the Compton-Getting effect, reproducing the spirit of Fig. 6/7 sector-2 vs sector-6 streaming.

**한국어.** 이 노트북은 합성 데이터로 EPAM의 핵심 개념 4가지를 구현한다:
1. 5개 SSD 망원경(LEMS30, LEMS120, LEFS60, LEFS150, CA60)의 각도 기하 — sector look-direction을 unit sphere에 표시.
2. 에너지와 stopping power($dE/dx$)에 따른 전자/이온 분리 — LEMS(자석)와 LEFS(foil) 설계 및 CA60 $\Delta E \times E$ 이온 종 식별의 물리적 토대.
3. 1일 baseline 위 $4\sigma$ trigger로 SEP onset을 감지하는 near-real-time logic — 1998년부터 NOAA RTSW SEP 경보를 지탱한 운영 알고리즘.
4. sector 카운트로부터의 first-order anisotropy와 Compton-Getting 효과 — 그림 6/7의 sector-2 vs sector-6 streaming 정신을 재현.

In [ ]:
"""Imports and global plotting configuration for the EPAM implementation notebook.

All cells use NumPy, SciPy, and Matplotlib only. No external data files are read;
every figure is reproduced from synthetic data designed to match the qualitative
behavior described in Gold et al. (1998).
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d

rng = np.random.default_rng(seed=19970825)  # ACE launch date as seed

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Part 1: 5 SSD Telescope Angular Geometry / 5개 SSD 망원경 각도 기하

**English.** EPAM's five telescopes are mounted at fixed polar angles ($\theta_{\mathrm{tel}}$) measured from the spacecraft spin axis. As ACE rotates with spin period $T_s = 12$ s, each telescope sweeps a great-circle band at constant $\theta_{\mathrm{tel}}$, and that band is divided into either 4 sectors (90 degrees each, dwell 3 s) or 8 sectors (45 degrees each, dwell 1.5 s). The combination of five polar bands and many azimuthal sectors yields nearly full unit-sphere coverage — unique among ACE instruments.

Look direction of the $k$-th sector at telescope $i$ in spacecraft frame:
$$\hat{n}_{i,k} = (\sin\theta_i\cos\phi_{i,k},\;\sin\theta_i\sin\phi_{i,k},\;\cos\theta_i)$$
where $\phi_{i,k} = (k - 0.5)\,\Delta\phi_i$ is the sector center azimuth.

**한국어.** EPAM의 다섯 망원경은 위성 spin 축으로부터 측정한 고정 polar angle ($\theta_{\mathrm{tel}}$)에 장착되어 있다. ACE가 spin 주기 $T_s = 12$ s로 회전하면 각 망원경은 일정한 $\theta_{\mathrm{tel}}$에서 대원(great circle) 띠를 sweep하고, 그 띠는 4 sector(90도, dwell 3 s) 또는 8 sector(45도, dwell 1.5 s)로 분할된다. 다섯 polar 띠와 다수의 azimuth sector의 조합으로 거의 전 unit-sphere를 커버 — ACE 기기 중 유일한 능력이다.

In [ ]:
"""Define the five EPAM telescope geometries.

Returns
-------
telescopes : list of dict
    Each entry has keys: name, theta_deg (polar angle from spin axis),
    n_sectors, dwell_s, geometric_factor_cm2sr, full_cone_deg, role.
"""

telescopes = [
    {'name': 'LEMS30',  'theta_deg':  30.0, 'n_sectors': 4, 'dwell_s': 3.0,
     'geometric_factor_cm2sr': 0.428, 'full_cone_deg': 51.0, 'role': 'ions'},
    {'name': 'LEFS60',  'theta_deg':  60.0, 'n_sectors': 8, 'dwell_s': 1.5,
     'geometric_factor_cm2sr': 0.397, 'full_cone_deg': 53.0, 'role': 'electrons'},
    {'name': 'CA60',    'theta_deg':  60.0, 'n_sectors': 8, 'dwell_s': 1.5,
     'geometric_factor_cm2sr': 0.103, 'full_cone_deg': 45.0, 'role': 'ion-composition'},
    {'name': 'LEMS120', 'theta_deg': 120.0, 'n_sectors': 8, 'dwell_s': 1.5,
     'geometric_factor_cm2sr': 0.428, 'full_cone_deg': 51.0, 'role': 'ions'},
    {'name': 'LEFS150', 'theta_deg': 150.0, 'n_sectors': 4, 'dwell_s': 3.0,
     'geometric_factor_cm2sr': 0.397, 'full_cone_deg': 53.0, 'role': 'electrons'},
]


def sector_look_directions(theta_deg: float, n_sectors: int) -> np.ndarray:
    """Compute unit-vector look directions of all sectors of a telescope.

    Args:
        theta_deg: Polar angle of the telescope axis (degrees from spin z-axis).
        n_sectors: Number of azimuthal sectors per spin (4 or 8 for EPAM).

    Returns:
        Array of shape (n_sectors, 3) with sector-center unit vectors in the
        spacecraft frame (z = spin axis, +x = arbitrary reference at phi = 0).
    """
    theta = np.deg2rad(theta_deg)
    dphi = 2.0 * np.pi / n_sectors
    phi = (np.arange(n_sectors) + 0.5) * dphi
    return np.column_stack([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.full_like(phi, np.cos(theta)),
    ])


print(f"{'Telescope':<10} {'theta':>6} {'sectors':>8} {'dwell(s)':>9} {'G(cm^2 sr)':>11}  role")
print('-' * 60)
for t in telescopes:
    print(f"{t['name']:<10} {t['theta_deg']:>5.0f}  {t['n_sectors']:>8d} "
          f"{t['dwell_s']:>9.1f} {t['geometric_factor_cm2sr']:>11.3f}  {t['role']}")

In [ ]:
"""Visualize the 5 telescope sector look directions on a 3-D unit sphere
and a Mollweide-style 2-D projection. The figure mirrors Gold et al. Fig. 2
but in our own (x, y, z) coordinates.
"""

colors = {'LEMS30': 'tab:red', 'LEFS60': 'tab:blue', 'CA60': 'tab:green',
          'LEMS120': 'tab:orange', 'LEFS150': 'tab:purple'}

fig = plt.figure(figsize=(13, 6))

# 3-D sphere with sector centers
ax3d = fig.add_subplot(1, 2, 1, projection='3d')
u = np.linspace(0, 2 * np.pi, 60)
v = np.linspace(0, np.pi, 30)
ax3d.plot_surface(np.outer(np.cos(u), np.sin(v)),
                  np.outer(np.sin(u), np.sin(v)),
                  np.outer(np.ones_like(u), np.cos(v)),
                  color='lightgray', alpha=0.15, edgecolor='none')
for t in telescopes:
    n = sector_look_directions(t['theta_deg'], t['n_sectors'])
    ax3d.scatter(n[:, 0], n[:, 1], n[:, 2], s=80, color=colors[t['name']],
                 label=f"{t['name']} ({t['n_sectors']} sec)", depthshade=True)
ax3d.quiver(0, 0, 0, 0, 0, 1.2, color='k', arrow_length_ratio=0.1)
ax3d.text(0, 0, 1.3, 'spin axis', fontsize=9)
ax3d.set_xlabel('x'); ax3d.set_ylabel('y'); ax3d.set_zlabel('z (spin)')
ax3d.set_title('EPAM sector look directions / sector look 방향')
ax3d.legend(loc='upper left', fontsize=8)

# 2-D theta-phi map (cylindrical projection)
ax2d = fig.add_subplot(1, 2, 2)
for t in telescopes:
    n = sector_look_directions(t['theta_deg'], t['n_sectors'])
    phi = np.rad2deg(np.arctan2(n[:, 1], n[:, 0])) % 360.0
    theta = np.rad2deg(np.arccos(n[:, 2]))
    ax2d.scatter(phi, theta, s=120, color=colors[t['name']],
                 edgecolor='k', label=t['name'])
    for i, (ph, th) in enumerate(zip(phi, theta)):
        ax2d.annotate(str(i + 1), (ph, th), fontsize=7, ha='center', va='center')
ax2d.invert_yaxis()
ax2d.set_xlabel('Azimuth phi (deg) / 방위각')
ax2d.set_ylabel('Polar angle theta (deg) / 극각')
ax2d.set_title('Cylindrical projection / 원통 투영')
ax2d.legend(loc='center right', fontsize=8)
ax2d.set_xlim(0, 360); ax2d.set_ylim(180, 0)

plt.tight_layout()
plt.show()

## Part 2: Electron / Ion Separation by Energy and dE/dx / 에너지와 dE/dx에 의한 전자·이온 분리

**English.** EPAM achieves clean electron/ion discrimination through two complementary techniques:

1. **LEFS foil** (0.35 mg/cm$^2$ aluminized Parylene): ions below $\sim$350 keV cannot punch through (large $dE/dx$), but electrons above $\sim$35 keV pass through (small $dE/dx$). The foil therefore acts as a low-pass filter for electrons and a high-pass filter for ions.
2. **LEMS magnet** (rare-earth, $B \approx 0.2$ T over short path): the Lorentz force deflects electrons below $\sim$500 keV out of the aperture but barely perturbs ions of the same energy (because the gyroradius $r_g = p / (qB)$ scales with momentum, and for equal kinetic energy, ion momentum is much larger).

**한국어.** EPAM은 보완적인 두 기법으로 깨끗한 전자/이온 분리를 달성한다:
1. **LEFS foil** (0.35 mg/cm$^2$ aluminized Parylene): 약 350 keV 이하 이온은 $dE/dx$가 커서 foil을 관통하지 못하고, 약 35 keV 이상 전자는 $dE/dx$가 작아 통과 — 전자에 대한 low-pass, 이온에 대한 high-pass 필터.
2. **LEMS 자석** (희토류, 짧은 경로에 $B \approx 0.2$ T): Lorentz 힘이 약 500 keV 이하 전자를 입구 밖으로 휘어내지만, 같은 에너지의 이온은 운동량이 훨씬 크므로 거의 영향을 받지 않음.

Below we model both: a simplified Bethe-Bloch-like $dE/dx \propto z^2 / \beta^2$ for ions, and the magnetic deflection radius for electrons. Then we reproduce the qualitative $\Delta E \times E$ matrix of CA60 (Fig. 10) showing distinct hyperbolic tracks for H, He, CNO, and Fe.

In [ ]:
"""Simplified non-relativistic stopping power dE/dx for an ion of charge z
and mass number A at kinetic energy E (MeV/nuc).

    -dE/dx ~ K * z^2 / beta^2  ~ K' * z^2 * A / E

We use natural-unit constants normalized so that protons at 1 MeV deposit
approximately 25 MeV/(g/cm^2) in silicon — a reasonable order-of-magnitude
match for the EPAM thin-D detector (4.8 micron) regime.
"""

MP_C2_MEV = 938.272  # proton rest energy [MeV]
ME_C2_MEV = 0.511    # electron rest energy [MeV]
K_BB = 0.307         # Bethe-Bloch coefficient [MeV cm^2 / g] approx
Z_OVER_A_SI = 14.0 / 28.086
I_SI_MEV = 173e-6


def stopping_power_ion(E_MeV_per_nuc: np.ndarray, z: int, A: int) -> np.ndarray:
    """Approximate non-relativistic stopping power for an ion in silicon.

    Args:
        E_MeV_per_nuc: Kinetic energy per nucleon (MeV/nuc).
        z: Ion charge state (assumed fully stripped).
        A: Mass number.

    Returns:
        Stopping power -dE/dx in MeV / (g/cm^2). Total energy loss in the
        4.8 micron thin D detector follows by multiplying by the column density.
    """
    E = np.asarray(E_MeV_per_nuc, dtype=float)
    gamma = 1.0 + E / MP_C2_MEV
    beta2 = 1.0 - 1.0 / gamma**2
    log_term = np.log(2.0 * ME_C2_MEV * beta2 * gamma**2 / I_SI_MEV) - beta2
    log_term = np.maximum(log_term, 1.0)  # floor at low energies
    return K_BB * Z_OVER_A_SI * (z**2) / np.maximum(beta2, 1e-3) * log_term


def gyroradius_electron(E_keV: np.ndarray, B_tesla: float = 0.2) -> np.ndarray:
    """Gyroradius (cm) of an electron of kinetic energy E_keV in field B.

    Uses fully relativistic momentum: p c = sqrt((E + m_e c^2)^2 - (m_e c^2)^2).

    Args:
        E_keV: Kinetic energy (keV).
        B_tesla: Magnetic flux density of the LEMS magnet (Tesla).

    Returns:
        Gyroradius in cm. The LEMS magnet path is ~3 cm; electrons whose
        gyroradius is below this value are deflected away from detector M.
    """
    E_MeV = np.asarray(E_keV) * 1e-3
    pc_MeV = np.sqrt((E_MeV + ME_C2_MEV)**2 - ME_C2_MEV**2)  # MeV
    p_kg_m_s = pc_MeV * 1.602e-13 / 2.998e8  # kg m/s
    r_m = p_kg_m_s / (1.602e-19 * B_tesla)
    return r_m * 100.0  # cm


# Sanity print
for E in [100.0, 500.0, 1000.0]:
    print(f"electron {E:5.0f} keV: r_gyro = {gyroradius_electron(E):.2f} cm "
          f"(deflected if << 3 cm)")
for sp, z, A in [('H', 1, 1), ('He', 2, 4), ('O', 8, 16), ('Fe', 26, 56)]:
    dEdx = stopping_power_ion(np.array([1.0]), z, A)[0]
    print(f"{sp:>2s} at 1 MeV/nuc: -dE/dx = {dEdx:7.1f} MeV cm^2/g  (z^2*A = {z**2 * A})")

In [ ]:
"""Reproduce the spirit of Gold et al. Fig. 10: a Delta-E vs E_residual matrix
for the CA60 telescope showing distinct hyperbolic tracks for H, He, CNO, Fe.

We simulate ~3000 ions sampled from a power-law spectrum, compute the energy
deposited in the thin D detector (4.8 micron Si = 1.12e-3 g/cm^2) and the
residual energy in the thick C detector (200 micron Si = 4.66e-2 g/cm^2),
then plot the resulting log-log scatter.
"""

D_THICKNESS_GCM2 = 4.8e-4 * 2.33  # 4.8 micron Si
C_THICKNESS_GCM2 = 200e-4 * 2.33  # 200 micron Si

species = [
    ('H',  1,  1, 1.00, 'tab:blue'),
    ('He', 2,  4, 0.10, 'tab:cyan'),
    ('O',  8, 16, 0.005, 'tab:green'),
    ('Fe', 26, 56, 0.001, 'tab:red'),
]

fig, ax = plt.subplots(figsize=(9, 7))
for sp, z, A, frac, color in species:
    n_events = int(3000 * frac)
    if n_events < 5:
        n_events = 50
    # Power-law sampling in MeV/nuc, gamma = 2.5
    u = rng.uniform(0, 1, n_events)
    E_per_nuc = 0.3 * (1 - u)**(-1.0 / 1.5)  # ~0.3 to ~10 MeV/nuc
    E_per_nuc = E_per_nuc[E_per_nuc < 20.0]
    dEdx = stopping_power_ion(E_per_nuc, z, A)
    delta_E = dEdx * D_THICKNESS_GCM2 * A    # MeV deposited in D (multiply by A for total)
    E_total_MeV = E_per_nuc * A
    E_residual = np.maximum(E_total_MeV - delta_E, 0.01)
    # Add detector resolution noise (5% Gaussian)
    delta_E *= rng.normal(1.0, 0.05, len(delta_E))
    E_residual *= rng.normal(1.0, 0.05, len(E_residual))
    ax.scatter(E_residual, delta_E, s=10, color=color, alpha=0.5, label=f"{sp} (z={z}, A={A})")

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('E_residual in C detector (MeV) / C 검출기 잔여에너지')
ax.set_ylabel('Delta-E in D detector (MeV) / D 검출기 에너지 손실')
ax.set_title('Synthetic CA60 Delta-E x E_residual matrix (cf. Gold 1998 Fig. 10)\n'
             '합성 CA60 Delta-E x E_residual 매트릭스')
ax.legend()
ax.set_xlim(0.05, 200); ax.set_ylim(0.001, 200)
plt.tight_layout()
plt.show()

print('Hyperbolic tracks separated by z^2*A factor confirm the basis of W1-W8 species channels.')

## Part 3: Near-Real-Time SEP Onset Detection / 근실시간 SEP onset 감지

**English.** EPAM channels feed NOAA's Real-Time Solar Wind (RTSW) system, which broadcasts SEP alerts when energetic particle counts rise significantly above quiet-time background. A widely used trigger condition is:

$$\text{ALERT if}\quad C(t) > \mu_{1\mathrm{day}}(t) + 4 \cdot \sigma_{1\mathrm{day}}(t)$$

where $\mu_{1\mathrm{day}}, \sigma_{1\mathrm{day}}$ are the rolling mean and standard deviation over a 1-day (1440-minute) window of count rates. Because energetic electrons travel near $c$ while ions are non-relativistic, the electron channel onset typically precedes the ion onset by 10-60 min — providing the operational lead time of the alert.

**한국어.** EPAM 채널은 NOAA의 RTSW 시스템에 공급되어, 정상시 background보다 에너지 입자 카운트가 유의미하게 상승할 때 SEP 경보를 송출한다. 널리 쓰이는 trigger 조건:

$$\text{ALERT}\quad C(t) > \mu_{1\mathrm{day}}(t) + 4 \sigma_{1\mathrm{day}}(t)$$

여기서 $\mu_{1\mathrm{day}}, \sigma_{1\mathrm{day}}$는 1일(1440분) 창의 rolling 평균과 표준편차. 에너지 전자는 $c$에 가까운 속도, 이온은 비상대론적이므로 보통 전자 채널 onset이 이온 onset보다 10-60분 앞서 발생 — 이 시간차가 경보의 운영 lead time을 만든다.

In [ ]:
"""Generate a synthetic 5-day SEP event time series at 1-min cadence.

Two channels:
  - DE2 (53-103 keV electrons, LEMS30 deflected-electron channel)
  - P3  (115-193 keV ions, LEMS30 ion channel)

Background is Poisson with quiet-time means typical of L1 (a few c/s).
An impulsive event injects a fast-rising / slow-decaying enhancement,
with the electron pulse arriving ~30 min before the ion pulse to mimic
velocity dispersion (the basis of the RTSW 'electron-leads-ion' lead time).
"""

n_minutes = 5 * 24 * 60                 # 5 days at 1-minute cadence
t = np.arange(n_minutes)                # minutes from start

bg_e = 5.0   # quiet-time electron count rate (c/s, channel-integrated)
bg_i = 8.0   # quiet-time ion count rate


def sep_pulse(t_min: np.ndarray, t_onset: float, peak: float, tau_rise: float,
              tau_decay: float) -> np.ndarray:
    """Asymmetric exponential rise/decay pulse for synthetic SEP injection."""
    dt = t_min - t_onset
    pulse = np.zeros_like(dt, dtype=float)
    rise = (dt >= 0) & (dt < 4 * tau_rise)
    pulse[rise] = peak * (1 - np.exp(-dt[rise] / tau_rise))
    decay = dt >= 4 * tau_rise
    peak_value = peak * (1 - np.exp(-4))
    pulse[decay] = peak_value * np.exp(-(dt[decay] - 4 * tau_rise) / tau_decay)
    return pulse


rate_e_true = bg_e + sep_pulse(t, t_onset=2 * 24 * 60 + 0,    peak=600.0,
                                tau_rise=20.0, tau_decay=12 * 60.0)
rate_i_true = bg_i + sep_pulse(t, t_onset=2 * 24 * 60 + 30,   peak=400.0,
                                tau_rise=40.0, tau_decay=20 * 60.0)

# Poisson counting noise per 60-second sample
counts_e = rng.poisson(rate_e_true * 60.0).astype(float) / 60.0
counts_i = rng.poisson(rate_i_true * 60.0).astype(float) / 60.0

print(f'simulated electron channel: bg={bg_e:.1f}, peak~{rate_e_true.max():.1f} c/s')
print(f'simulated ion channel    : bg={bg_i:.1f}, peak~{rate_i_true.max():.1f} c/s')

In [ ]:
"""Compute the 4-sigma above 1-day baseline trigger and visualize lead time.

Algorithm (per channel):
  1. Compute rolling mean mu and rolling std sigma over the previous
     1440 samples (= 1 day at 1-minute cadence).
  2. ALERT if C(t) > mu(t) + 4 * sigma(t) for a sample.
  3. Onset time = first ALERT sample within the event window.

The 1-day baseline is computed *causally* (uses only past samples) so that
the algorithm matches what an operational pipeline can do at time t.
"""

WINDOW = 1440  # 1 day in minutes
K_SIGMA = 4.0


def rolling_baseline(x: np.ndarray, window: int) -> tuple[np.ndarray, np.ndarray]:
    """Causal rolling mean and std over a trailing window.

    Args:
        x: 1-D time series.
        window: Number of trailing samples to use.

    Returns:
        Tuple (mu, sigma) of arrays the same length as x. The first window-1
        samples are filled with NaN since no full baseline is available.
    """
    mu = np.full_like(x, np.nan, dtype=float)
    sd = np.full_like(x, np.nan, dtype=float)
    for i in range(window, len(x)):
        seg = x[i - window:i]
        mu[i] = seg.mean()
        sd[i] = seg.std(ddof=1)
    return mu, sd


mu_e, sd_e = rolling_baseline(counts_e, WINDOW)
mu_i, sd_i = rolling_baseline(counts_i, WINDOW)
thr_e = mu_e + K_SIGMA * sd_e
thr_i = mu_i + K_SIGMA * sd_i

alert_e = counts_e > thr_e
alert_i = counts_i > thr_i

# First alert minute in each channel (onset)
onset_e = np.argmax(alert_e) if alert_e.any() else -1
onset_i = np.argmax(alert_i) if alert_i.any() else -1
lead_min = onset_i - onset_e if (onset_e > 0 and onset_i > 0) else 0

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, counts, thr, onset, label, color in [
    (axes[0], counts_e, thr_e, onset_e, '53-103 keV electrons (DE2)', 'tab:red'),
    (axes[1], counts_i, thr_i, onset_i, '115-193 keV ions (P3)',     'tab:blue'),
]:
    ax.plot(t / 60.0, counts, color=color, lw=0.7, label=label)
    ax.plot(t / 60.0, thr, color='k', lw=1.0, ls='--',
            label=f'mu + {K_SIGMA:.0f} sigma threshold')
    if onset > 0:
        ax.axvline(onset / 60.0, color='lime', lw=2.0,
                   label=f'first alert at t = {onset / 60.0:.2f} h')
    ax.set_yscale('log')
    ax.set_ylabel('count rate (c/s) / 카운트율')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_ylim(0.5, 5e3)
axes[1].set_xlabel('time (hours from start) / 시간')
axes[0].set_title(f'EPAM-style 4-sigma SEP onset detection / 4-sigma SEP onset 감지\n'
                  f'electron-leads-ion lead time = {lead_min} min')
plt.tight_layout()
plt.show()

print(f'electron onset: minute {onset_e}  ({onset_e / 60.0:.2f} h)')
print(f'ion onset    : minute {onset_i}  ({onset_i / 60.0:.2f} h)')
print(f'lead time    : {lead_min} minutes (electrons before ions)')

## Part 4: Anisotropy from Sector Counts (Compton-Getting) / sector 카운트로부터의 anisotropy

**English.** EPAM's azimuthal sectoring lets us compute the **first-order anisotropy** of the particle flux. For a power-law spectrum $j(E) \propto E^{-\gamma}$ and a streaming source, the count rate in a sector pointing along unit vector $\hat{n}_k$ is

$$C_k = C_0 \,(1 + A_1 \,\hat{n}_k \cdot \hat{s})$$

where $\hat{s}$ is the streaming direction (typically along the local Parker spiral) and $A_1$ is the first-order anisotropy amplitude. The **Compton-Getting** correction relates the convective component to the bulk flow $\vec{V}$ of the medium:

$$A_{1,\mathrm{CG}} = (2 + \gamma) \frac{V}{v}\cos\theta_{V}$$

where $v$ is particle speed and $\theta_V$ the angle between $\vec V$ and look direction.

We invert: from 8 LEMS120 sector counts $C_k$, fit $C_0$, $A_1$, and the streaming azimuth $\phi_s$ via least squares — the same exercise an EPAM data analyst does to characterize anisotropic events like Fig. 6 sub-event B (sectors 2-4 high, 7-8 quiet).

**한국어.** EPAM의 azimuth sector 분할은 입자 플럭스의 **first-order anisotropy**를 계산할 수 있게 한다. 멱법칙 스펙트럼 $j(E) \propto E^{-\gamma}$과 streaming source에서 sector $k$의 카운트율은

$$C_k = C_0 \,(1 + A_1 \,\hat{n}_k \cdot \hat{s})$$

여기서 $\hat{s}$는 streaming 방향(보통 Parker 나선), $A_1$은 first-order anisotropy 진폭. **Compton-Getting** 보정은 매질의 bulk flow $\vec V$와 관련된 대류 성분:

$$A_{1,\mathrm{CG}} = (2 + \gamma)\frac{V}{v}\cos\theta_V$$

여기서 $v$는 입자 속도, $\theta_V$는 $\vec V$와 look 방향의 각도. 8 sector LEMS120 카운트로부터 $C_0, A_1, \phi_s$를 최소제곱으로 역산 — 그림 6 부 이벤트 B(sector 2-4 강, 7-8 약) 같은 anisotropic 이벤트를 특성화할 때 EPAM 분석가가 수행하는 작업과 동일하다.

In [ ]:
"""Generate two synthetic LEMS120 sector-count snapshots (8 sectors at theta=120 deg)
and fit the first-order anisotropy model C_k = C0 (1 + A1 cos(phi_k - phi_s)).

Snapshot B: strong streaming along phi_s = 90 deg (intense in sectors 2-4).
Snapshot D: nearly isotropic plateau (later in the event).
These mirror Gold et al. 1998 Fig. 6 sub-events B and D.
"""

n_sectors = 8
phi_centers = np.deg2rad((np.arange(n_sectors) + 0.5) * (360.0 / n_sectors))


def synth_sector_counts(C0: float, A1: float, phi_s_deg: float,
                        n_sec: int = 8, total_time_s: float = 12.0) -> np.ndarray:
    """Synthesize Poisson-noised sector counts for one spin.

    Args:
        C0: Mean count rate (c/s, sector-averaged).
        A1: First-order anisotropy amplitude (0 = isotropic, ~1 = strong).
        phi_s_deg: Streaming direction azimuth (degrees).
        n_sec: Number of sectors.
        total_time_s: Spin period to convert rate to total counts per sector.

    Returns:
        Counts per sector for a single spin (Poisson realizations).
    """
    phi_s = np.deg2rad(phi_s_deg)
    phi = (np.arange(n_sec) + 0.5) * (2.0 * np.pi / n_sec)
    expected_rate = C0 * (1.0 + A1 * np.cos(phi - phi_s))
    expected_counts = np.maximum(expected_rate, 0.1) * (total_time_s / n_sec)
    return rng.poisson(expected_counts).astype(float)


C0_true_B, A1_true_B, phi_s_true_B = 600.0, 0.85, 90.0
C0_true_D, A1_true_D, phi_s_true_D = 250.0, 0.05, 0.0

counts_B = synth_sector_counts(C0_true_B, A1_true_B, phi_s_true_B)
counts_D = synth_sector_counts(C0_true_D, A1_true_D, phi_s_true_D)
rates_B = counts_B / (12.0 / n_sectors)  # convert back to c/s per sector
rates_D = counts_D / (12.0 / n_sectors)


def fit_first_order(rates: np.ndarray, phi: np.ndarray) -> tuple[float, float, float]:
    """Linear least-squares fit of C(phi) = C0 + a cos(phi) + b sin(phi).

    Args:
        rates: Sector count rates (c/s).
        phi: Sector center azimuths (radians).

    Returns:
        Tuple (C0_fit, A1_fit, phi_s_fit_deg) recovering the model parameters.
    """
    X = np.column_stack([np.ones_like(phi), np.cos(phi), np.sin(phi)])
    coef, *_ = np.linalg.lstsq(X, rates, rcond=None)
    C0_fit, a, b = coef
    A1_fit = np.hypot(a, b) / C0_fit
    phi_s_fit = np.rad2deg(np.arctan2(b, a)) % 360.0
    return C0_fit, A1_fit, phi_s_fit


fitB = fit_first_order(rates_B, phi_centers)
fitD = fit_first_order(rates_D, phi_centers)
print(f'Snapshot B (anisotropic): true C0={C0_true_B:.0f}, A1={A1_true_B:.2f}, '
      f'phi_s={phi_s_true_B:.0f} deg')
print(f'                fit:      C0={fitB[0]:.0f}, A1={fitB[1]:.2f}, '
      f'phi_s={fitB[2]:.0f} deg')
print(f'Snapshot D (isotropic)  : true C0={C0_true_D:.0f}, A1={A1_true_D:.2f}')
print(f'                fit:      C0={fitD[0]:.0f}, A1={fitD[1]:.2f}, '
      f'phi_s={fitD[2]:.0f} deg')

In [ ]:
"""Compton-Getting illustration and pie-plot visualization of the two snapshots.

Pie plots in the spin plane (Fig. 7 in Gold 1998) show sector-resolved rates.
We also compute the Compton-Getting prediction A_CG = (2+gamma) V/v cos(theta_V)
for typical solar wind parameters.
"""

# Compton-Getting numerical prediction
gamma_spec = 2.5                         # SEP power-law slope
V_sw_kms = 450.0                         # solar wind speed (km/s)
E_part_keV = 100.0                       # 100 keV ion
v_part_kms = 2.998e5 * np.sqrt(2.0 * E_part_keV / 1e3 / MP_C2_MEV)
A_CG_max = (2.0 + gamma_spec) * V_sw_kms / v_part_kms
print(f'Compton-Getting maximum A1: ({2 + gamma_spec:.1f}) * {V_sw_kms:.0f} / {v_part_kms:.0f} = '
      f'{A_CG_max:.3f}')
print(f'  (i.e. ~{A_CG_max * 100:.1f}% modulation from solar wind convection alone)')

fig, axes = plt.subplots(1, 2, figsize=(13, 6), subplot_kw={'projection': 'polar'})
phi_smooth = np.linspace(0, 2 * np.pi, 360)
for ax, rates, fit, title in [
    (axes[0], rates_B, fitB, 'Snapshot B: anisotropic streaming / 강한 anisotropy'),
    (axes[1], rates_D, fitD, 'Snapshot D: nearly isotropic / 거의 isotropic'),
]:
    width = 2 * np.pi / n_sectors
    ax.bar(phi_centers, rates, width=width * 0.95,
           color='tab:orange', edgecolor='k', alpha=0.7,
           label='sector rates / sector 카운트율')
    C0_f, A1_f, phi_s_f = fit
    model = C0_f * (1 + A1_f * np.cos(phi_smooth - np.deg2rad(phi_s_f)))
    ax.plot(phi_smooth, model, color='tab:blue', lw=2.0,
            label=f'fit: A1={A1_f:.2f}, phi_s={phi_s_f:.0f} deg')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize=8, bbox_to_anchor=(1.25, -0.05))
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(1)

plt.tight_layout()
plt.show()

print('\nPie plots reproduce the spirit of Gold et al. 1998 Fig. 7:')
print('  - Snapshot B: strong sectoral imbalance, fitted streaming direction recovered.')
print('  - Snapshot D: near-uniform pie, A1 << 0.1 consistent with isotropy.')

## Summary / 요약

**English.** This notebook captured four pillars of the EPAM design and operations:

| Concept / 개념 | Implementation here / 본 노트북 구현 | Modern equivalent / 현대 동등물 |
|---|---|---|
| 5 SSD telescope geometry | `sector_look_directions()` returns unit vectors for 30/60/120/150 deg polar angles | Solar Orbiter EPD (EPT/HET/STEP), PSP ISOIS-EPI-Lo/Hi |
| Electron/ion separation | `stopping_power_ion()` $z^2 A / E$ scaling + `gyroradius_electron()` for magnet | $\Delta E \times E$ Si telescopes still standard in 2026 |
| 4-sigma SEP onset trigger | Causal rolling baseline over 1440 minutes, alert when count > $\mu + 4\sigma$ | NOAA SWPC SEP alerts (still EPAM-driven 24/7); ESA Vigil baseline |
| First-order anisotropy fit | Linear LSQ of $C_0 + a\cos\phi + b\sin\phi$ over 8 sectors | PSP ISOIS pitch-angle distributions; SO/EPD anisotropy products |

**한국어.** 이 노트북은 EPAM 설계와 운영의 네 기둥을 다루었다: (1) 5개 SSD 망원경 기하 — 30/60/120/150 deg polar angle의 단위 벡터; (2) 전자/이온 분리 — $z^2 A / E$ 스케일의 stopping power와 자석에 의한 전자 휘어짐; (3) 4-sigma SEP onset trigger — 1440분 인과적 rolling baseline 기반의 NOAA SWPC 경보 알고리즘; (4) first-order anisotropy 적합 — 8 sector 카운트의 선형 최소제곱으로 streaming 방향 복원 + Compton-Getting 보정. 이 네 가지는 EPAM이 1997년 가동을 시작한 이후 2026년 현재까지 약 29년간 L1 우주환경 baseline으로 기능해온 실질적 토대를 코드로 재현한 것이다.